# Chess Learning — Stage 02 (Stockfish Teacher): TD Learning vs Stockfish

Instead of self-play, the engine learns by playing games against Stockfish at a fixed ELO.
The engine alternates colours across games (odd game numbers = White, even = Black).

| Component | Description |
|---|---|
| **Opponent** | Stockfish at configurable ELO (minimum 1320 for Stockfish 19) |
| **TD(0) learning** | Identical to `selfplay_td0.ipynb` — weights updated after every game |
| **No epsilon-greedy** | Stockfish selects its own moves; engine always plays its best move |
| **SQLite logging** | Same tables as self-play; compatible with all R analysis scripts |

**Research question:** Does a calibrated external teacher produce faster or more correct
weight convergence than self-play? Compare weight trajectories against run_id=14/15.


In [35]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'chess', '-q'])

0

In [36]:
import asyncio
import chess
import chess.engine
import chess.pgn
import glob
import math
import sqlite3
import time
import os
from datetime import datetime
from collections import defaultdict

### Import engine components

In **Positron**, `engine.py` is imported from the same folder.


In [37]:
sys.path.insert(0, os.path.dirname(os.path.abspath('selfplay_sf.ipynb')))
from engine import FEATURE_NAMES, default_weights, extract_features, get_best_move
print('Engine loaded. Features:', FEATURE_NAMES)

Engine loaded. Features: ['passed_pawn', 'doubled_pawn', 'isolated_pawn', 'backward_pawn', 'king_safety', 'center_control', 'rook_open_file', 'connected_rooks', 'knight_pst', 'bishop_pair', 'rook_seventh', 'piece_development', 'mobility', 'pawn_advancement']


## Database Setup

Identical schema to `selfplay_td0.ipynb` — all R analysis scripts work unchanged.


In [38]:
def setup_database(db_path):
    os.makedirs(os.path.dirname(os.path.abspath(db_path)), exist_ok=True)
    conn = sqlite3.connect(db_path)
    conn.executescript('''
        CREATE TABLE IF NOT EXISTS runs (
            run_id        INTEGER PRIMARY KEY,
            started_at    TEXT,
            depth         INTEGER,
            learning_rate REAL,
            move_limit    INTEGER,
            notes         TEXT
        );
        CREATE TABLE IF NOT EXISTS games (
            game_id       INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id        INTEGER,
            game_number   INTEGER,
            outcome       TEXT,
            n_halfmoves   INTEGER,
            duration_s    REAL,
            terminated_by TEXT
        );
        CREATE TABLE IF NOT EXISTS weights (
            weight_id     INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id        INTEGER,
            game_number   INTEGER,
            feature_name  TEXT,
            weight_value  REAL
        );
        CREATE TABLE IF NOT EXISTS feature_stats (
            stat_id       INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id        INTEGER,
            game_number   INTEGER,
            outcome       TEXT,
            feature_name  TEXT,
            mean_value    REAL
        );
        CREATE TABLE IF NOT EXISTS game_records (
            record_id     INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id        INTEGER,
            game_number   INTEGER,
            pgn           TEXT,
            td_error      REAL
        );
        CREATE TABLE IF NOT EXISTS weight_deltas (
            delta_id      INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id        INTEGER,
            game_number   INTEGER,
            feature_name  TEXT,
            delta         REAL
        );
        CREATE TABLE IF NOT EXISTS position_logs (
            pos_id        INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id        INTEGER,
            game_number   INTEGER,
            ply           INTEGER,
            color         TEXT,
            feature_name  TEXT,
            feature_value REAL
        );
        CREATE TABLE IF NOT EXISTS stockfish_evals (
            eval_id      INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id       INTEGER,
            game_number  INTEGER,
            sf_elo       INTEGER,
            n_games      INTEGER,
            wins         INTEGER,
            losses       INTEGER,
            draws        INTEGER,
            win_rate     REAL,
            source       TEXT
        );
    ''')
    conn.commit()
    try:
        conn.execute('ALTER TABLE runs ADD COLUMN last_updated TEXT')
        conn.commit()
    except sqlite3.OperationalError:
        pass
    return conn


def log_run(conn, run_id, depth, learning_rate, move_limit, notes=''):
    conn.execute(
        'INSERT OR REPLACE INTO runs (run_id, started_at, depth, learning_rate, move_limit, notes) '
        'VALUES (?, ?, ?, ?, ?, ?)',
        (run_id, datetime.now().isoformat(), depth, learning_rate, move_limit, notes)
    )
    conn.commit()


def log_game(conn, run_id, game_number, outcome, n_halfmoves, duration_s, terminated_by):
    conn.execute(
        'INSERT INTO games (run_id,game_number,outcome,n_halfmoves,duration_s,terminated_by) '
        'VALUES (?,?,?,?,?,?)',
        (run_id, game_number, outcome, n_halfmoves, duration_s, terminated_by)
    )
    conn.commit()


def log_weights(conn, run_id, game_number, weights):
    rows = [(run_id, game_number, name, val) for name, val in weights.items()]
    conn.executemany(
        'INSERT INTO weights (run_id,game_number,feature_name,weight_value) VALUES (?,?,?,?)',
        rows
    )
    conn.commit()


def log_feature_stats(conn, run_id, game_number, positions, outcome):
    if not positions:
        return
    sums = {name: 0.0 for name in FEATURE_NAMES}
    for features, color in positions:
        sign = 1.0 if color == chess.WHITE else -1.0
        for name, val in features.items():
            sums[name] += sign * val
    n = len(positions)
    rows = [(run_id, game_number, outcome, name, sums[name] / n) for name in FEATURE_NAMES]
    conn.executemany(
        'INSERT INTO feature_stats (run_id,game_number,outcome,feature_name,mean_value) VALUES (?,?,?,?,?)',
        rows
    )
    conn.commit()


def load_last_weights(conn, run_id):
    max_game = conn.execute(
        'SELECT MAX(game_number) FROM weights WHERE run_id = ?', (run_id,)
    ).fetchone()[0]
    if max_game is None:
        raise ValueError(f'No weight checkpoints found for run_id={run_id}')
    rows = conn.execute(
        'SELECT feature_name, weight_value FROM weights '
        'WHERE run_id = ? AND game_number = ?',
        (run_id, max_game)
    ).fetchall()
    return {name: val for name, val in rows}


def get_last_game_number(conn, run_id):
    result = conn.execute(
        'SELECT MAX(game_number) FROM games WHERE run_id = ?', (run_id,)
    ).fetchone()[0]
    return result if result is not None else 0


def log_game_record(conn, run_id, game_number, pgn, td_error):
    conn.execute(
        'INSERT INTO game_records (run_id,game_number,pgn,td_error) VALUES (?,?,?,?)',
        (run_id, game_number, pgn, td_error)
    )
    conn.commit()


def log_weight_deltas(conn, run_id, game_number, deltas):
    rows = [(run_id, game_number, name, delta) for name, delta in deltas.items()]
    conn.executemany(
        'INSERT INTO weight_deltas (run_id,game_number,feature_name,delta) VALUES (?,?,?,?)',
        rows
    )
    conn.commit()


def log_position_logs(conn, run_id, game_number, positions):
    rows = []
    for ply, (features, color) in enumerate(positions):
        color_str = 'white' if color == chess.WHITE else 'black'
        for name, val in features.items():
            rows.append((run_id, game_number, ply, color_str, name, val))
    conn.executemany(
        'INSERT INTO position_logs (run_id,game_number,ply,color,feature_name,feature_value) '
        'VALUES (?,?,?,?,?,?)',
        rows
    )
    conn.commit()

## Stockfish Setup

Auto-detects Stockfish installed via `winget install Stockfish.Stockfish`.


In [39]:
def find_stockfish():
    local_app = os.environ.get('LOCALAPPDATA', '')
    pattern   = os.path.join(
        local_app, 'Microsoft', 'WinGet', 'Packages', 'Stockfish*', '**', 'stockfish*.exe'
    )
    hits = glob.glob(pattern, recursive=True)
    if hits:
        return hits[0]
    raise FileNotFoundError(
        'Stockfish not found. Install with: winget install Stockfish.Stockfish'
    )

SF_PATH = find_stockfish()
print(f'Stockfish found: {SF_PATH}')

# Windows: Jupyter uses SelectorEventLoop which doesn't support subprocess transport.
# Switch to ProactorEventLoop so chess.engine.SimpleEngine.popen_uci() works.
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    print('Windows ProactorEventLoop set.')

Stockfish found: C:\Users\dkill\AppData\Local\Microsoft\WinGet\Packages\Stockfish.Stockfish_Microsoft.Winget.Source_8wekyb3d8bbwe\stockfish\stockfish-windows-x86-64-universal.exe
Windows ProactorEventLoop set.


## Game Function

Each call plays one complete game. Our engine uses negamax at `DEPTH`; Stockfish
uses its own search at the configured ELO. All positions (features + side to move)
are recorded for the TD update, regardless of who made the move.


In [40]:
def play_game_vs_sf(sf, weights, depth, engine_color, move_limit=80):
    """
    Play one game: our engine (engine_color) vs Stockfish.
    sf: open SimpleEngine, pre-configured with UCI_Elo limit.
    Returns: positions, outcome, terminated_by, pgn_str
    """
    board         = chess.Board()
    game          = chess.pgn.Game()
    node          = game
    positions     = []          # list of (features_dict, color_to_move)
    terminated_by = 'natural'

    while True:
        if board.is_game_over():
            terminated_by = 'natural'
            break
        if len(positions) >= move_limit:
            terminated_by = 'move_limit'
            break

        features = extract_features(board)
        color    = board.turn

        if board.turn == engine_color:
            move, _ = get_best_move(board, weights, depth=depth, randomize=False)
        else:
            result = sf.play(board, chess.engine.Limit(time=0.5), ponder=False)
            move   = result.move

        if move is None:
            break

        node = node.add_variation(move)
        board.push(move)
        positions.append((features, color))

    outcome = '1/2-1/2' if terminated_by == 'move_limit' else board.result()
    game.headers['Result'] = outcome
    return positions, outcome, terminated_by, str(game)

## TD(0) Learning

Identical to `selfplay_td0.ipynb`. The learning signal is unchanged — only the
source of opponent moves differs.


In [41]:
def outcome_for_color(outcome, color):
    if outcome == '1-0':
        return  1.0 if color == chess.WHITE else -1.0
    elif outcome == '0-1':
        return -1.0 if color == chess.WHITE else  1.0
    return 0.0


def td_update(weights, positions, outcome, learning_rate, draw_lr_scales=None):
    """
    TD(0) update: target for position t is -V(s_{t+1}), except the terminal
    position which uses the actual game outcome.
    """
    is_draw   = (outcome == '1/2-1/2')
    dr_scales = draw_lr_scales or {}
    values = [
        sum(weights.get(name, 0.0) * val for name, val in features.items())
        for features, color in positions
    ]
    weights_before = dict(weights)
    total_error    = 0.0
    for t, (features, color) in enumerate(positions):
        target = -values[t + 1] if t < len(positions) - 1 else outcome_for_color(outcome, color)
        error  = target - values[t]
        total_error += abs(error)
        for name, val in features.items():
            effective_lr = learning_rate * dr_scales.get(name, 1.0) if is_draw else learning_rate
            weights[name] += effective_lr * error * val
    for name in weights:
        weights[name] = max(-50.0, min(50.0, weights[name]))
    deltas   = {name: weights[name] - weights_before[name] for name in weights}
    td_error = total_error / len(positions) if positions else 0.0
    return deltas, td_error

## ELO Estimation

After training, `estimate_elo()` plays a fixed evaluation batch against Stockfish
using the final weights (no weight updates). The result is converted to an ELO point
estimate and 95% confidence interval via the standard ELO formula:

$$\hat{E} = E_{SF} + 400 \cdot \log_{10}\!\left(\frac{S}{1-S}\right)$$

where $$S$$ is the observed score (win=1, draw=0.5, loss=0) over the evaluation games.
Results are logged to `stockfish_evals` so they appear in `06_stockfish_arena.R`.


In [42]:
def estimate_elo(sf, weights, depth, sf_elo, n_eval=20, move_limit=80):
    """
    Estimate engine ELO from n_eval games vs Stockfish at sf_elo.
    No weight updates. Returns (elo_est, elo_lo, elo_hi, wins, losses, draws, score).
    elo_lo / elo_hi are approximate 95% confidence bounds.
    """
    wins = losses = draws = 0
    for i in range(n_eval):
        engine_color = chess.WHITE if i % 2 == 0 else chess.BLACK
        board  = chess.Board()
        n_half = 0
        while not board.is_game_over() and n_half < move_limit:
            if board.turn == engine_color:
                move, _ = get_best_move(board, weights, depth=depth, randomize=False)
            else:
                move = sf.play(board, chess.engine.Limit(time=0.5), ponder=False).move
            if move is None:
                break
            board.push(move)
            n_half += 1
        outcome = board.result()
        engine_won  = (outcome == '1-0' and engine_color == chess.WHITE) or \
                      (outcome == '0-1' and engine_color == chess.BLACK)
        engine_lost = (outcome == '0-1' and engine_color == chess.WHITE) or \
                      (outcome == '1-0' and engine_color == chess.BLACK)
        if   engine_won:  wins   += 1
        elif engine_lost: losses += 1
        else:             draws  += 1

    score = (wins + 0.5 * draws) / n_eval

    # Clip to avoid log(0) — treat 0% or 100% as floor/ceiling
    score_c = max(0.001, min(0.999, score))
    elo_est = sf_elo + 400 * math.log10(score_c / (1 - score_c))

    # 95% CI via normal approximation on score
    se   = math.sqrt(score_c * (1 - score_c) / n_eval)
    lo_c = max(0.001, score_c - 1.96 * se)
    hi_c = min(0.999, score_c + 1.96 * se)
    elo_lo = sf_elo + 400 * math.log10(lo_c / (1 - lo_c))
    elo_hi = sf_elo + 400 * math.log10(hi_c / (1 - hi_c))

    return round(elo_est), round(elo_lo), round(elo_hi), wins, losses, draws, score


def log_sf_eval(db_path, run_id, game_number, sf_elo, n_eval, wins, losses, draws):
    win_rate = (wins + 0.5 * draws) / n_eval if n_eval > 0 else 0.0
    conn = sqlite3.connect(db_path)
    conn.execute(
        'INSERT INTO stockfish_evals '
        '(run_id,game_number,sf_elo,n_games,wins,losses,draws,win_rate,source) '
        'VALUES (?,?,?,?,?,?,?,?,?)',
        (run_id, game_number, sf_elo, n_eval, wins, losses, draws, win_rate, 'sf_teacher')
    )
    conn.commit()
    conn.close()

## Training Loop

Opens one Stockfish process for the full session and closes it safely via `try/finally`.
Engine alternates colours: odd game numbers = White, even = Black.


In [43]:
def train_vs_sf(
    run_id,
    n_games,
    sf_elo           = 1320,
    depth            = 3,
    learning_rate    = 0.001,
    move_limit       = 80,
    checkpoint_every = 10,
    db_path          = '../../data/chess_learning.db',
    notes            = '',
    resume           = False,
    log_positions    = False,
    weight_init      = 'zero',
    draw_lr_scales   = None,
    verbose          = True,
):
    conn = setup_database(db_path)

    if resume:
        weights    = load_last_weights(conn, run_id)
        start_game = get_last_game_number(conn, run_id) + 1
        conn.execute('UPDATE runs SET last_updated = ? WHERE run_id = ?',
                     (datetime.now().isoformat(), run_id))
        conn.commit()
    else:
        weights    = default_weights(weight_init)
        start_game = 1
        log_run(conn, run_id, depth, learning_rate, move_limit, notes)
        log_weights(conn, run_id, 0, weights)

    outcomes = {'1-0': 0, '0-1': 0, '1/2-1/2': 0}
    t_start  = time.time()

    sf = chess.engine.SimpleEngine.popen_uci(SF_PATH)
    sf.configure({'UCI_LimitStrength': True, 'UCI_Elo': sf_elo})
    print(f'Stockfish open at ELO {sf_elo}. Starting run_id={run_id} from game {start_game}.')

    try:
        for i in range(n_games):
            game_num     = start_game + i
            engine_color = chess.WHITE if (game_num % 2 == 1) else chess.BLACK
            color_label  = 'White' if engine_color == chess.WHITE else 'Black'
            t0           = time.time()

            positions, outcome, terminated_by, pgn = play_game_vs_sf(
                sf, weights, depth, engine_color, move_limit
            )
            elapsed = time.time() - t0

            deltas, td_error = td_update(
                weights, positions, outcome, learning_rate, draw_lr_scales
            )
            log_game(conn, run_id, game_num, outcome, len(positions), elapsed, terminated_by)
            log_game_record(conn, run_id, game_num, pgn, td_error)
            log_weight_deltas(conn, run_id, game_num, deltas)
            log_feature_stats(conn, run_id, game_num, positions, outcome)
            if log_positions:
                log_position_logs(conn, run_id, game_num, positions)
            outcomes[outcome] = outcomes.get(outcome, 0) + 1

            if game_num % checkpoint_every == 0:
                log_weights(conn, run_id, game_num, weights)
                if verbose:
                    elapsed_total = time.time() - t_start
                    rate = (i + 1) / elapsed_total * 3600
                    print(
                        f'Game {game_num:5d} ({color_label}) | {outcome:7s} | '
                        f'{len(positions):3d} half-moves | {elapsed:.1f}s | '
                        f'td_err={td_error:.3f} | {rate:.0f} games/hr'
                    )
    finally:
        sf.quit()
        print('Stockfish closed.')

    final_game = start_game + n_games - 1
    log_weights(conn, run_id, final_game, weights)
    conn.close()

    total_time = time.time() - t_start
    if verbose:
        action = 'resumed' if resume else 'complete'
        print(f'\nRun {run_id} {action}: {n_games} games in {total_time/60:.1f} min '
              f'(games {start_game} to {final_game})')
        print(f'Outcomes: {outcomes}')
        print(f'Database: {os.path.abspath(db_path)}')

    return weights

## Configuration & Run

Set `RESUME = False` to start a new run; `RESUME = True` to continue.
`RUN_ID = 16` is reserved for the Stockfish-teacher variant.

**Typical session sizes:**
- Quick check: `N_GAMES = 20` (~30–60 min at ELO 1320, depth 3)
- Medium session: `N_GAMES = 100` (~3–5 hr)

> **Speed note:** Stockfish games run ~60–120s each (Stockfish uses 0.5s per move).


In [44]:
RUN_ID           = 16
N_GAMES          = 500          # games to play in this session
N_EVAL           = 30          # evaluation games for ELO estimation (no weight updates)
SF_ELO           = 1320        # Stockfish ELO (minimum 1320 for Stockfish 19)
DEPTH            = 3
LEARNING_RATE    = 0.001
MOVE_LIMIT       = 80          # half-moves; shorter than self-play
CHECKPOINT_EVERY = 10
DB_PATH          = '../../data/chess_learning.db'
WEIGHT_INIT      = 'zero'
RESUME           = True
LOG_POSITIONS    = False       # disabled by default; SF games are slower
DRAW_LR_SCALES   = {'king_safety': 0.05, 'center_control': 0.2}

final_weights = train_vs_sf(
    run_id           = RUN_ID,
    n_games          = N_GAMES,
    sf_elo           = SF_ELO,
    depth            = DEPTH,
    learning_rate    = LEARNING_RATE,
    move_limit       = MOVE_LIMIT,
    checkpoint_every = CHECKPOINT_EVERY,
    db_path          = DB_PATH,
    resume           = RESUME,
    log_positions    = LOG_POSITIONS,
    weight_init      = WEIGHT_INIT,
    draw_lr_scales   = DRAW_LR_SCALES,
    notes            = (
        f'Stockfish teacher; SF_ELO={SF_ELO}; 14 features; hard-coded material; '
        f'draw_lr king_safety=0.05 center_control=0.2; '
        f'depth={DEPTH}, lr={LEARNING_RATE}, init={WEIGHT_INIT}'
    ),
)

Stockfish open at ELO 1320. Starting run_id=16 from game 61.
Game    70 (Black) | 1/2-1/2 |  80 half-moves | 59.3s | td_err=0.023 | 56 games/hr
Game    80 (Black) | 1/2-1/2 |  80 half-moves | 108.5s | td_err=0.022 | 53 games/hr
Game    90 (Black) | 0-1     |  26 half-moves | 16.5s | td_err=0.061 | 58 games/hr
Game   100 (Black) | 1/2-1/2 |  80 half-moves | 85.0s | td_err=0.022 | 56 games/hr
Game   110 (Black) | 1/2-1/2 |  80 half-moves | 105.3s | td_err=0.035 | 56 games/hr
Game   120 (Black) | 1/2-1/2 |  80 half-moves | 73.5s | td_err=0.033 | 56 games/hr
Game   130 (Black) | 1/2-1/2 |  80 half-moves | 64.0s | td_err=0.035 | 55 games/hr
Game   140 (Black) | 1/2-1/2 |  80 half-moves | 56.1s | td_err=0.027 | 53 games/hr
Game   150 (Black) | 1/2-1/2 |  80 half-moves | 65.2s | td_err=0.026 | 53 games/hr
Game   160 (Black) | 1-0     |  59 half-moves | 49.6s | td_err=0.038 | 54 games/hr
Game   170 (Black) | 1/2-1/2 |  80 half-moves | 55.5s | td_err=0.032 | 54 games/hr
Game   180 (Black) | 1/2

## Post-Training ELO Estimate

Plays `N_EVAL` evaluation games with the final weights (no learning). Reports ELO
point estimate and 95% CI. Logged to `stockfish_evals` with `source = 'sf_teacher'`.


In [45]:
sf_eval = chess.engine.SimpleEngine.popen_uci(SF_PATH)
sf_eval.configure({'UCI_LimitStrength': True, 'UCI_Elo': SF_ELO})

try:
    elo_est, elo_lo, elo_hi, e_wins, e_losses, e_draws, score = estimate_elo(
        sf_eval, final_weights, DEPTH, SF_ELO, n_eval=N_EVAL, move_limit=MOVE_LIMIT
    )
finally:
    sf_eval.quit()

# Determine last game number for logging
conn = sqlite3.connect(DB_PATH)
last_game = get_last_game_number(conn, RUN_ID)
conn.close()

log_sf_eval(DB_PATH, RUN_ID, last_game, SF_ELO, N_EVAL, e_wins, e_losses, e_draws)

print(f'ELO estimate:  {elo_est}  (95% CI: {elo_lo} – {elo_hi})')
print(f'Eval record:   {e_wins}W / {e_losses}L / {e_draws}D  '
      f'(score={score:.3f} over {N_EVAL} games vs Stockfish {SF_ELO})')
print(f'Logged to stockfish_evals for run_id={RUN_ID}, game={last_game}')

ELO estimate:  1212  (95% CI: 1056 – 1334)
Eval record:   3W / 12L / 15D  (score=0.350 over 30 games vs Stockfish 1320)
Logged to stockfish_evals for run_id=16, game=560


## Weight Trajectory

Quick check after training. Richer visualization in the R scripts.


In [46]:
conn = sqlite3.connect(DB_PATH)
rows = conn.execute(
    'SELECT game_number, feature_name, weight_value FROM weights '
    'WHERE run_id = ? ORDER BY game_number, feature_name',
    (RUN_ID,)
).fetchall()
conn.close()

by_game = defaultdict(dict)
for game_num, feature, val in rows:
    by_game[game_num][feature] = val

game_nums = sorted(by_game.keys())
header = f"{'Feature':<24}" + ''.join(f'{g:>10}' for g in game_nums)
print(header)
print('-' * len(header))
for feature in FEATURE_NAMES:
    row = f'{feature:<24}' + ''.join(f"{by_game[g].get(feature, 0):>10.4f}" for g in game_nums)
    print(row)

Feature                          0        10        20        30        40        50        60        70        80        90       100       110       120       130       140       150       160       170       180       190       200       210       220       230       240       250       260       270       280       290       300       310       320       330       340       350       360       370       380       390       400       410       420       430       440       450       460       470       480       490       500       510       520       530       540       550       560
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------